# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [21]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import ollama
import json
import requests
from typing import List 
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [22]:
# Initialisation de l'objet qui va utiliser les endpoint de OpenAI
client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

MODEL = "llama3.2:1b"

In [23]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [11]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [24]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [13]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [25]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [15]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/


In [26]:
def get_links(url):
    website = Website(url)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [33]:
# Anthropic a fait des changements sur leur site qui rendent le scraping plus difficile, donc j'utilise HuggingFace à la place..
url_company = "https://mistral.ai"
nom_company = "Mistral"

company = Website(url_company)
company.links

['/',
 '/pricing',
 'https://console.mistral.ai/',
 '/contact',
 '/',
 '/products/vibe',
 '/contact',
 'http://console.mistral.ai',
 '/customers',
 '/customers',
 '/customers/stellantis',
 '/customers#asml',
 '/customers#cma-cgm',
 '/products/le-chat',
 '/products/vibe',
 'https://console.mistral.ai/',
 '/services',
 '/partners',
 '/careers',
 'https://console.mistral.ai',
 '/contact',
 '/',
 'https://apps.apple.com/us/app/le-chat-by-mistral-ai/id6740410176',
 'https://play.google.com/store/apps/details?id=ai.mistral.chat',
 '/about',
 '/customers',
 '/careers',
 '/contact',
 '/solutions',
 '/partners',
 '/news?category=research',
 'https://docs.mistral.ai/',
 '/products/studio',
 '/products/le-chat',
 '/products/vibe',
 '/products/compute',
 'https://legal.mistral.ai/terms',
 'https://legal.mistral.ai/terms/privacy-policy?language=en-US',
 'https://legal.mistral.ai/terms/data-processing-addendum',
 '/legal',
 '/brand',
 'https://x.com/mistralai',
 'https://www.linkedin.com/company/mis

In [25]:
get_links(url_company)

{'links': [{'type': 'about', 'url': 'https://console.mistral.ai/about'},
  {'type': 'careers', 'url': 'https://console.mistral.ai/careers'},
  {'type': 'products/studio',
   'url': 'https://legal.mistral.ai/products/studio'},
  {'type': 'brand', 'url': 'https://x.com/mistralai'},
  {'type': 'about', 'url': 'https://www.linkedin.com/company/mistralai/'},
  {'type': 'services', 'url': 'https://console.mistral.ai/services'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [34]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [27]:
print(get_all_details(url_company))

Found links: {'links': [{'type': 'about page', 'url': 'https://mistral.ai/about'}, {'type': 'careers page', 'url': 'https://mistral.ai/careers'}, {'type': 'customers/stellantis', 'url': 'https://mistral.ai/customers#stellantis'}, {'type': 'products/compute', 'url': 'https://mistral.ai/products/compute'}, {'type': 'products/vibe', 'url': 'https://mistral.ai/products/vibe'}]}
Landing page:
Webpage Title:
Frontier AI LLMs, assistants, agents, services | Mistral AI
Webpage Contents:
Products
Solutions
Research
Resources
Pricing
Company
Try Studio
Talk to sales
New: Ship code with Mistral Vibe.
Frontier AI.
In your hands.
We help organizations build tailored AI systems
to solve the world’s hardest problems.
Get in touch
Start building
Meet our customers
Your AI future belongs in your hands.
Frontier intelligence, customized to you.
Make your AI your own. Train, distill, fine-tune, and build with state-of-the-art open source models.
Enterprise agents with deep context.
Deploy agents that exe

In [35]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [37]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
get_brochure_user_prompt(nom_company, url_company)

Found links: {'links': [{'type': 'careers', 'url': 'https://console.mistral.ai'}, {'type': 'products', 'url': 'https://console.mistral.ai/products/'}, {'type': 'about', 'url': 'https://app.box.com/mistralai/about'}]}


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'You are looking at a company called: Mistral\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nFrontier AI LLMs, assistants, agents, services | Mistral AI\nWebpage Contents:\nProducts\nSolutions\nResearch\nResources\nPricing\nCompany\nTry Studio\nTalk to sales\nNew: Ship code with Mistral Vibe.\nFrontier AI.\nIn your hands.\nWe help organizations build tailored AI systems\nto solve the world’s hardest problems.\nGet in touch\nStart building\nMeet our customers\nYour AI future belongs in your hands.\nFrontier intelligence, customized to you.\nMake your AI your own. Train, distill, fine-tune, and build with state-of-the-art open source models.\nEnterprise agents with deep context.\nDeploy agents that execute, adapt, and deliver real results—with powerful orchestration, tooling, and safety.\nSelf-contained private deployments.\nBuild privately anywhere—on-premises

In [38]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [32]:
create_brochure(nom_company, url_company)

NameError: name 'nom_company' is not defined

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [40]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [41]:
stream_brochure(nom_company, url_company)

Found links: {'links': [{'type': 'home page about', 'url': 'https://mistral.ai/about'}, {'type': 'products studio', 'url': 'https://mistral.ai/products/studio'}, {'type': 'products vibe', 'url': 'https://mistral.ai/products/vibe'}, {'type': 'services partners', 'url': 'https://mistral.ai/services/partners'}]}


# Mistral AI: Unlocking the Power of Frontier AI

### About Us

Mistral AI is a pioneering French artificial intelligence startup founded in April 2023 by three visionary researchers: Arthur Mensch, Guillaume Lample, and Timothée Lacroix. Our mission is to democratize artificial intelligence through open-source, efficient, and innovative products and solutions.

**Mission Statement**

We believe in a future where AI is abundant and accessible, empowering the world to build with—and benefit from—the most significant technology of our time. We strive to challenge the opaque-box nature of 'big AI' and make this cutting-edge technology accessible to all.

### Our Story

Our journey began when we assembled a team of expert researchers from Google DeepMind and Meta, united by their shared academic roots at École Polytechnique. We envisioned a different approach to artificial intelligence—to focus on empowering the world rather than just creating machines.

From our early days as a research organization, we have continued to innovate and push the boundaries of what is possible with AI. Today, Mistral AI seeks to build tailored AI systems for organizations to tackle their most complex challenges.

### Meet Our Customers

Our customers are at the heart of everything we do. We've worked with:

* *Stellantis*, accelerating automotive innovation with our Enterprise Agents
* *CMA CGM*, streamlining global maritime operations through AI-driven solutions
* Various other industries, empowering them to build more efficient and effective AI-powered systems

### Our Products

At Mistral AI, we offer a range of products that cater to diverse customer needs:

* **Frontier AI**: A suite of open-source AI models for deployment in production
* **Le Chat**: An AI-powered chatbot for better customer experience and personalized support
* **Vibe**: A platform for building custom AI applications with post-train models, endpoints, and more

### Our Solutions

Our team is dedicated to providing expert guidance throughout the application development lifecycle:

- **Custom Model Training**: Scaling and training AI models from scratch using domain-specific knowledge
- **End-to-End Observability**: Delivers detailed data flows for optimizing system performance and security
- **Deep Use Case Exploration**: Engages with users to explore business problems and identify potential solutions

### Our Partnerships

Mistral AI is supported by various organizations, including:

* *AWS*, leveraging their expertise in building scalable and secure AI deployments
* *Microsoft*, providing access to cutting-edge Azure services for AI development and deployment

To learn more, visit our **About us** page or explore our products and solutions in **Resources**.

### Pricing

Mistral AI offers various pricing options and plans tailored to diverse customer needs:

- Standard: Perfect for small-scale projects with minimal requirements
- Enterprise: Designed for larger organizations and enterprises requiring custom solutions
- Production: Support provided for production-grade deployments.

### Get in Touch

Reach out to our team of experts at sales@mistral.ai or explore the latest open roles on our careers page as an AI enthusiast.

**Contact Us**

About us | Mistral AI
| --- | ---


### Terms of Service and Privacy Policy

Privacy policy: We respect your privacy. Read our terms of service and data processing agreement to learn more.

Terms of service: Please review our comprehensive guidelines for understanding how we collect data, use data, and protect it in accordance with applicable laws.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>

# TESTONS GRADIO AVEC NOTRE PROJET DE BROCHURE POUR LES ENTREPRISE 


In [42]:
import gradio as gr

In [43]:
def stream_ollama(company_name, url):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result
    

In [45]:
stream_ollama("LLM Engeneer Edward Donner", "https://edwarddonner.com")

<generator object stream_ollama at 0x734545056680>

In [47]:
view = gr.Interface(
    fn=stream_ollama,
    title="OLLAMA",
    inputs=[gr.Textbox(label="Nom de la companie:"), gr.Textbox(label="URL de la companie:")],
    outputs=[gr.Markdown(label="Response:")],
    examples=[["LLM Engeneer Edward Donner", "https://edwarddonner.com"]],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Found links: {'links': [{'type': 'about page', 'url': 'https://www.edwarddonner.com/about-me-and-about-nebula/'}, {'type': 'posts', 'url': 'https://edwarddonner.com/posts/'}, {'type': 'news', 'url': 'https://news.ycombinator.com'}]}
